In [ ]:
# Cell 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Imports
import os
import gc
import glob
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import scipy.io
from scipy.ndimage import gaussian_filter
from tqdm import tqdm
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# Cell 3: Dataset Paths
DATA_PATH = "/content/drive/MyDrive/Metro-Crowd-Project/datasets/ShanghaiTech"

print("Dataset root exists:", os.path.exists(DATA_PATH))

TRAIN_A = os.path.join(DATA_PATH, "part_A", "train_data", "images")
TEST_A  = os.path.join(DATA_PATH, "part_A", "test_data",  "images")

print("Train A images:", len(os.listdir(TRAIN_A)))
print("Test A images: ", len(os.listdir(TEST_A)))

In [ ]:
# Cell 4: Density Map Helpers
def load_points(mat_path):
    mat    = scipy.io.loadmat(mat_path)
    points = mat["image_info"][0][0][0][0][0]
    return points

def generate_density_map(image, points, sigma=4):
    h, w    = image.shape[:2]
    density = np.zeros((h, w), dtype=np.float32)
    for p in points:
        x = min(w - 1, int(p[0]))
        y = min(h - 1, int(p[1]))
        density[y, x] = 1
    density = gaussian_filter(density, sigma)
    # Normalize: total sum = number of people
    n_people = len(points)
    if density.sum() > 0:
        density = density / density.sum() * n_people
    return density

In [ ]:
# Cell 5: Dataset Class (pre-cached)
class ShanghaiTechDataset(Dataset):
    def __init__(self, image_folder, transform=None):
        self.transform = transform
        self.data = []

        image_paths = sorted(glob.glob(os.path.join(image_folder, "*.jpg")))
        gt_paths = []
        for img_path in image_paths:
            filename  = os.path.basename(img_path)
            mat_name  = "GT_" + filename.replace(".jpg", ".mat")
            gt_folder = os.path.join(
                os.path.dirname(os.path.dirname(img_path)), "ground-truth"
            )
            gt_paths.append(os.path.join(gt_folder, mat_name))

        print("Pre-loading dataset into RAM...")
        for img_path, gt_path in tqdm(zip(image_paths, gt_paths), total=len(image_paths)):
            img     = cv2.imread(img_path)
            img     = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            points  = load_points(gt_path)
            density = generate_density_map(img, points)

            # Resize once, not every epoch
            img     = cv2.resize(img, (512, 512))
            density = cv2.resize(density, (512, 512))

            # Re-normalize after resize to preserve total count
            n_people = len(points)
            if density.sum() > 0:
                density = density / density.sum() * n_people

            density = torch.tensor(density).unsqueeze(0)
            self.data.append((img, density))

        print(f"Loaded {len(self.data)} images into RAM.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img, density = self.data[idx]
        if self.transform:
            img = self.transform(img)
        return img, density

In [ ]:
# Cell 6: Transforms & DataLoaders
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std= [0.229, 0.224, 0.225]),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])

transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std= [0.229, 0.224, 0.225]),
])

train_dataset = ShanghaiTechDataset(TRAIN_A, transform=transform_train)
test_dataset  = ShanghaiTechDataset(TEST_A,  transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,
                          num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False,
                          num_workers=0)

print(f"Train: {len(train_dataset)} images | Test: {len(test_dataset)} images")
print(f"Train batches: {len(train_loader)}")

In [ ]:
# Cell 7: CSRNet Model
class CSRNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Frontend: VGG16 first 10 conv layers (pretrained, unfrozen)
        vgg = models.vgg16(weights='IMAGENET1K_V1')
        self.frontend = nn.Sequential(*list(vgg.features.children())[:23])

        # MLP Bridge: 1x1 convs = channel-wise MLP
        self.mlp = nn.Sequential(
            nn.Conv2d(512, 512, 1), nn.ReLU(),
            nn.Conv2d(512, 512, 1), nn.ReLU(),
            nn.Dropout2d(0.5),
            nn.Conv2d(512, 512, 1), nn.ReLU(),
        )

        # Backend: Dilated convolutions
        self.backend = nn.Sequential(
            nn.Conv2d(512, 256, 3, dilation=2, padding=2), nn.ReLU(),
            nn.Conv2d(256, 128, 3, dilation=2, padding=2), nn.ReLU(),
            nn.Conv2d(128, 64,  3, dilation=2, padding=2), nn.ReLU(),
            nn.Conv2d(64,  1,   1)
        )

    def forward(self, x):
        x = self.frontend(x)
        x = self.mlp(x)
        x = self.backend(x)
        return F.relu(x)

model = CSRNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params:,}")

In [ ]:
# Cell 8: Loss Function
def density_loss(pred, gt):
    # Downsample gt to match pred size (CSRNet outputs ~1/8 resolution)
    gt_small = F.interpolate(gt, size=pred.shape[2:], mode='bilinear', align_corners=False)

    mse        = F.mse_loss(pred, gt_small)
    pred_count = pred.sum(dim=[1, 2, 3])
    gt_count   = gt_small.sum(dim=[1, 2, 3])
    count_loss = F.l1_loss(pred_count, gt_count)

    # Higher count weight forces model to learn actual counts
    return mse + 0.1 * count_loss

In [ ]:
# Cell 9: Evaluation Function
def evaluate(model, loader):
    model.eval()
    total_mae = 0.0
    with torch.no_grad():
        for imgs, gt_density in loader:
            imgs       = imgs.to(device)
            gt_density = gt_density.to(device).float()
            pred       = model(imgs)

            # Upsample pred back to gt size for fair count comparison
            pred_up    = F.interpolate(pred, size=gt_density.shape[2:],
                                       mode='bilinear', align_corners=False)
            pred_count = pred_up.sum().item()
            gt_count   = gt_density.sum().item()

            total_mae += abs(pred_count - gt_count)

    return total_mae / len(loader)

In [ ]:
# Cell 10: Training Loop (with checkpoint resume)
def train(model, train_loader, test_loader, num_epochs=100,
          save_path="/content/best_csrnet.pth",
          checkpoint_path="/content/checkpoint.pth"):

    optimizer   = torch.optim.Adam(model.parameters(), lr=1e-5)
    scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=8, factor=0.5)
    scaler      = torch.amp.GradScaler('cuda')
    best_mae    = float('inf')
    history     = {"train_loss": [], "val_mae": []}
    start_epoch = 1

    # Resume from checkpoint if exists
    if os.path.exists(checkpoint_path):
        print("Resuming from checkpoint...")
        ckpt        = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        history     = ckpt['history']
        start_epoch = ckpt['epoch'] + 1
        best_mae    = min(history['val_mae']) if history['val_mae'] else float('inf')
        print(f"Resumed from epoch {ckpt['epoch']} | Best MAE: {best_mae:.2f}")

    for epoch in range(start_epoch, num_epochs + 1):

        # Train
        model.train()
        total_loss = 0.0
        for imgs, gt_density in tqdm(train_loader, desc=f"Epoch {epoch:03d}", leave=False):
            imgs       = imgs.to(device)
            gt_density = gt_density.to(device).float()

            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                pred = model(imgs)
                loss = density_loss(pred, gt_density)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        # Cleanup once per epoch
        torch.cuda.empty_cache()
        gc.collect()

        avg_loss = total_loss / len(train_loader)
        history["train_loss"].append(avg_loss)

        # Validate
        mae = evaluate(model, test_loader)
        history["val_mae"].append(mae)
        scheduler.step(mae)

        print(f"Epoch {epoch:03d} | Loss: {avg_loss:.4f} | MAE: {mae:.2f}", end="")

        if mae < best_mae:
            best_mae = mae
            torch.save(model.state_dict(), save_path)
            print(f"  ✅ Best (MAE={best_mae:.2f})", end="")

        if epoch % 5 == 0:
            torch.save({
                'epoch':     epoch,
                'model':     model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'history':   history
            }, checkpoint_path)
            print(f"  💾 Checkpoint saved", end="")

        print()

    print(f"\nDone. Best MAE: {best_mae:.2f}")
    return history

In [ ]:
# Cell 11: Run Training
history = train(
    model,
    train_loader,
    test_loader,
    num_epochs=100,
    save_path="/content/best_csrnet.pth",
    checkpoint_path="/content/checkpoint.pth"
)

In [ ]:
# Cell 12: Plot Results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history["train_loss"], label="Train Loss")
ax1.set_title("Training Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(history["val_mae"], label="Val MAE", color="orange")
ax2.axhline(y=100, color='r', linestyle='--', label="Target MAE=100")
ax2.set_title("Validation MAE")
ax2.set_xlabel("Epoch")
ax2.legend()

plt.tight_layout()
plt.show()
print(f"Best MAE achieved: {min(history['val_mae']):.2f}")

In [ ]:
# Cell 13: Visualize Predictions
def visualize(model, dataset, n=3):
    model.eval()
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    axes[0][0].set_title("Image")
    axes[0][1].set_title("GT Density")
    axes[0][2].set_title("Predicted Density")

    mean = torch.tensor([0.485, 0.456, 0.406])
    std  = torch.tensor([0.229, 0.224, 0.225])

    with torch.no_grad():
        for i in range(n):
            img, gt = dataset[i]
            pred    = model(img.unsqueeze(0).to(device))
            pred_up = F.interpolate(pred, size=gt.shape[1:],
                                    mode='bilinear', align_corners=False)

            img_show = img.permute(1, 2, 0) * std + mean
            img_show = img_show.clamp(0, 1).numpy()

            pred_count = pred_up.sum().item()
            gt_count   = gt.sum().item()

            axes[i][0].imshow(img_show)
            axes[i][0].set_title(f"Image {i+1}")
            axes[i][1].imshow(gt.squeeze(), cmap='jet')
            axes[i][1].set_title(f"GT Count: {gt_count:.1f}")
            axes[i][2].imshow(pred_up.squeeze().cpu(), cmap='jet')
            axes[i][2].set_title(f"Pred Count: {pred_count:.1f}")
            for ax in axes[i]:
                ax.axis('off')

    plt.tight_layout()
    plt.show()

model.load_state_dict(torch.load("/content/best_csrnet.pth", map_location=device))
visualize(model, test_dataset, n=3)